In [1]:
#| default_exp jepa

# 01 · The JEPA world model

> The model itself: how it encodes, predicts, imagines the future, and scores plans. Exports to `lewm/jepa.py`.

[Notebook 00](00_module.ipynb) built the parts. This one wires them into a
world model.

## The one idea

A video prediction model usually predicts **pixels**. That is wasteful: to
predict the next frame exactly, the model must model every shadow, texture and
bit of noise, most of which does not matter for deciding what to do.

A **J**oint **E**mbedding **P**redictive **A**rchitecture predicts in
**embedding space** instead:

```
        pixels ──encoder──▶ embedding ──┐
                                        ├──predictor──▶ predicted next embedding
        action ──encoder──▶ embedding ──┘                        │
                                                                 │ compare
        next pixels ──encoder──▶ true next embedding ────────────┘
```

The model never reconstructs an image. It only has to get the *embedding*
right, and it gets to choose what an embedding means — so it can throw away
everything irrelevant.

The catch is the one from notebook 00: if the encoder is free to choose, the
easiest choice is a constant. That is what `SIGReg` prevents.

## The four parts

| Part | Job | Shape |
|:---|:---|:---|
| `encoder` | a ViT: image → vector | `(B*T, C, H, W) → (B*T, 192)` |
| `projector` | MLP head on the encoder | `(B*T, 192) → (B*T, 192)` |
| `action_encoder` | `Embedder`: raw action → vector | `(B, T, 10) → (B, T, 192)` |
| `predictor` | `ARPredictor`: dynamics | `(B, T, 192) → (B, T, 192)` |

In [2]:
#| export
"""JEPA Implementation"""

import torch
import torch.nn.functional as F
from einops import rearrange
from torch import nn

## A small helper

Used when snapshotting a dict of tensors during rollout. `detach()` cuts the
tensor out of the autograd graph, while `clone()` gives it its own memory so
later in-place writes cannot corrupt the snapshot.

In [3]:
#| export
def detach_clone(v: torch.Tensor) -> torch.Tensor:
    """Return a detached, independent copy of a tensor."""
    return v.detach().clone()

## The model

Note `projector` and `pred_proj` default to `nn.Identity()`. That is a neat
trick: the model works with or without the projection heads, and the code
downstream never needs to check which.

The flow through `encode` is worth reading slowly, because the reshaping is the
part that confuses people:

```
info["pixels"]        (B, T, C, H, W)   a batch of short video clips
rearrange             (B*T, C, H, W)    the ViT has no concept of time, so
                                        every frame becomes an independent image
encoder               (B*T, tokens, D)  ViT output, one vector per image patch
[:, 0]                (B*T, D)          keep only the CLS token: the ViT's
                                        whole-image summary vector
projector             (B*T, D)
rearrange             (B, T, D)         fold time back out
```

In [4]:
#| export
class JEPA(nn.Module):
    """Joint Embedding Predictive Architecture.

    Args:
        encoder: image -> tokens. A ViT; only its CLS token is used.
        predictor: ARPredictor. (emb, act_emb) -> next emb.
        action_encoder: Embedder. raw actions -> embeddings.
        projector: optional MLP head after the encoder.
        pred_proj: optional MLP head after the predictor.
    """

    def __init__(
        self,
        encoder: nn.Module,
        predictor: nn.Module,
        action_encoder: nn.Module,
        projector: nn.Module | None = None,
        pred_proj: nn.Module | None = None,
    ) -> None:
        super().__init__()

        self.encoder = encoder
        self.predictor = predictor
        self.action_encoder = action_encoder
        # Identity keeps the call sites uniform when no head is configured.
        self.projector = projector or nn.Identity()
        self.pred_proj = pred_proj or nn.Identity()

    def encode(
        self, info: dict[str, torch.Tensor]
    ) -> dict[str, torch.Tensor]:
        """Encode observations and actions into embeddings.
        info: dict with pixels and action keys

        Reads  info["pixels"]  (B, T, C, H, W)
               info["action"]  (B, T, A)          [optional]
        Writes info["emb"]     (B, T, D)
               info["act_emb"] (B, T, D)          [if action present]

        Mutates and returns the same dict.
        """

        pixels = info['pixels'].float()
        b = pixels.size(0)

        # The ViT sees images, not videos: fold time into the batch so every
        # frame is encoded independently.
        pixels = rearrange(pixels, "b t ... -> (b t) ...") # flatten for encoding

        # interpolate_pos_encoding lets the ViT accept an image size that
        # differs from the one its position embeddings were built for.
        output = self.encoder(pixels, interpolate_pos_encoding=True)

        # The CLS token is the ViT's summary of the whole image; patch tokens
        # are dropped. (B*T, D)
        pixels_emb = output.last_hidden_state[:, 0]  # cls token

        emb = self.projector(pixels_emb)
        info["emb"] = rearrange(emb, "(b t) d -> b t d", b=b)   # time back out

        if "action" in info:
            info["act_emb"] = self.action_encoder(info["action"])

        return info

    def predict(
        self, emb: torch.Tensor, act_emb: torch.Tensor
    ) -> torch.Tensor:
        """Predict next state embedding
        emb: (B, T, D)
        act_emb: (B, T, A_emb)
        returns: (B, T, D) -- the predicted NEXT embedding for each timestep
        """
        preds = self.predictor(emb, act_emb)

        # pred_proj is an MLP with BatchNorm, which needs a flat batch axis,
        # so time is folded in and back out around it.
        preds = self.pred_proj(rearrange(preds, "b t d -> (b t) d"))
        preds = rearrange(preds, "(b t) d -> b t d", b=emb.size(0))
        return preds

    ####################
    ## Inference only ##
    ####################

    def rollout(
        self,
        info: dict[str, torch.Tensor],
        action_sequence: torch.Tensor,
        history_size: int = 3,
    ) -> dict[str, torch.Tensor]:
        """Rollout the model given an initial info dict and action sequence.
        pixels: (B, S, T, C, H, W)
        action_sequence: (B, S, T, action_dim)
         - S is the number of action plan samples
         - T is the time horizon

        This is the model *imagining*. It encodes the real starting frames
        once, then repeatedly feeds its own predictions back in, never looking
        at a real image again. Writes info["predicted_emb"] (B, S, T', D).
        """

        assert "pixels" in info, "pixels not in info_dict"
        H = info["pixels"].size(2)
        B, S, T = action_sequence.shape[:3]

        # Split the plan: the first H actions accompany the real observed
        # frames, the rest are the future to imagine through.
        act_0, act_future = torch.split(action_sequence, [H, T - H], dim=2)
        info["action"] = act_0
        n_steps = T - H

        # Encode the real starting frames once. [:, 0] because every one of the
        # S candidate plans starts from the same observation.
        _init = {k: v[:, 0] for k, v in info.items() if torch.is_tensor(v)}
        _init = self.encode(_init)
        # expand (not repeat): a read-only broadcast view, no memory copied.
        emb = info["emb"] = _init["emb"].unsqueeze(1).expand(B, S, -1, -1)
        _init = {k: detach_clone(v) for k, v in _init.items()}

        # Treat the S plans as extra batch entries: one big parallel rollout.
        # .clone() because expand() gave a view and we are about to cat onto it.
        emb = rearrange(emb, "b s ... -> (b s) ...").clone()
        act = rearrange(act_0, "b s ... -> (b s) ...")
        act_future = rearrange(act_future, "b s ... -> (b s) ...")

        # rollout predictor autoregressively for n_steps
        HS = history_size
        for t in range(n_steps):
            act_emb = self.action_encoder(act)
            # Only the last HS steps are fed in: the predictor's position
            # embedding is sized for history_size, and it is Markovian enough.
            emb_trunc = emb[:, -HS:]  # (BS, HS, D)
            act_trunc = act_emb[:, -HS:]  # (BS, HS, A_emb)
            # [:, -1:] keeps only the newest prediction, as a length-1 slice.
            pred_emb = self.predict(emb_trunc, act_trunc)[:, -1:]  # (BS, 1, D)
            # The prediction becomes the next input: this is the imagination.
            emb = torch.cat([emb, pred_emb], dim=1)  # (BS, T+1, D)

            next_act = act_future[:, t : t + 1, :]  # (BS, 1, action_dim)
            act = torch.cat([act, next_act], dim=1)  # (BS, T+1, action_dim)

        # predict the last state
        act_emb = self.action_encoder(act)  # (BS, T, A_emb)
        emb_trunc = emb[:, -HS:]  # (BS, HS, D)
        act_trunc = act_emb[:, -HS:]  # (BS, HS, A_emb)
        pred_emb = self.predict(emb_trunc, act_trunc)[:, -1:]  # (BS, 1, D)
        emb = torch.cat([emb, pred_emb], dim=1)

        # unflatten batch and sample dimensions
        pred_rollout = rearrange(emb, "(b s) ... -> b s ...", b=B, s=S)
        info["predicted_emb"] = pred_rollout

        return info

    def criterion(self, info_dict: dict[str, torch.Tensor]) -> torch.Tensor:
        """Compute the cost between predicted embeddings and goal embeddings.

        Only the FINAL step is scored: a plan is judged by where it ends up,
        not by the path it took. Returns (B, S) -- one cost per candidate plan.
        """
        pred_emb = info_dict["predicted_emb"]  # (B,S, T-1, dim)
        goal_emb = info_dict["goal_emb"]  # (B, S, T, dim)

        goal_emb = goal_emb[..., -1:, :].expand_as(pred_emb)

        # return last-step cost per action candidate
        cost = F.mse_loss(
            pred_emb[..., -1:, :],
            goal_emb[..., -1:, :].detach(),
            reduction="none",
        ).sum(dim=tuple(range(2, pred_emb.ndim)))  # (B, S)

        return cost

    def get_cost(
        self,
        info_dict: dict[str, torch.Tensor],
        action_candidates: torch.Tensor,
    ) -> torch.Tensor:
        """ Compute the cost of action candidates given an info dict with goal and initial state.

        This is the planning entry point. A planner (CEM, Adam, ...) proposes
        S candidate action sequences; this scores all of them in one batch, and
        the planner keeps the best. Returns (B, S).
        """

        assert "goal" in info_dict, "goal not in info_dict"

        device = next(self.parameters()).device
        for k in list(info_dict.keys()):
            if torch.is_tensor(info_dict[k]):
                info_dict[k] = info_dict[k].to(device)

        # Build an info dict for the goal image and encode it the same way as
        # an observation -- the goal is just another frame.
        goal = {k: v[:, 0] for k, v in info_dict.items() if torch.is_tensor(v)}
        goal["pixels"] = goal["goal"]

        # Strip the "goal_" prefix so encode() sees the keys it expects.
        for k in info_dict:
            if k.startswith("goal_"):
                goal[k[len("goal_") :]] = goal.pop(k)

        goal.pop("action")
        goal = self.encode(goal)

        info_dict["goal_emb"] = goal["emb"]
        info_dict = self.rollout(info_dict, action_candidates)

        cost = self.criterion(info_dict)

        return cost

## Checking the pieces on real shapes

A full `JEPA` needs a ViT, which notebook 03 builds with the pretrained
weights. Here we check the parts that do not need one, using a stand-in
encoder so the shape logic can be verified in isolation.

In [5]:
# The notebooks live in nbs/, so the repo root has to be on sys.path for
# `import lewm.module` to resolve. Harmless when it is already there.
import sys, pathlib
_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "nbs" else pathlib.Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from typing import Any

from lewm.module import ARPredictor, Embedder, MLP

class FakeEncoder(torch.nn.Module):
    """Stands in for the ViT: returns the right shape, learns nothing.

    A real ViT returns an object with .last_hidden_state of shape
    (N, num_tokens, D); JEPA only ever reads token 0.
    """
    def __init__(self, d: int = 192) -> None:
        super().__init__()
        self.proj = torch.nn.Linear(3 * 224 * 224, d)

    def forward(
        self, x: torch.Tensor, interpolate_pos_encoding: bool = False
    ) -> Any:
        flat = x.flatten(1)                       # (N, 3*224*224)
        cls = self.proj(flat).unsqueeze(1)        # (N, 1, D)
        return type("Out", (), {"last_hidden_state": cls})()

D, A = 192, 10
model = JEPA(
    encoder=FakeEncoder(D),
    predictor=ARPredictor(num_frames=3, depth=2, heads=4, mlp_dim=256,
                          input_dim=D, hidden_dim=D, output_dim=D),
    action_encoder=Embedder(input_dim=A, smoothed_dim=A, emb_dim=D),
    projector=MLP(input_dim=D, hidden_dim=256, output_dim=D),
    pred_proj=MLP(input_dim=D, hidden_dim=256, output_dim=D),
).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"toy model parameters: {n_params/1e6:.1f}M")

toy model parameters: 30.3M


### `encode`: a clip of frames becomes a sequence of vectors

In [6]:
B, T = 2, 4
info = {
    "pixels": torch.rand(B, T, 3, 224, 224),   # a batch of short clips
    "action": torch.randn(B, T, A),
}

with torch.no_grad():
    out = model.encode(info)

print(f"pixels  {tuple(info['pixels'].shape)}  ->  emb     {tuple(out['emb'].shape)}")
print(f"action  {str(tuple(info['action'].shape)):18s}  ->  act_emb {tuple(out['act_emb'].shape)}")
print("\nencode() mutates the dict in place and returns it:", out is info)

pixels  (2, 4, 3, 224, 224)  ->  emb     (2, 4, 192)
action  (2, 4, 10)          ->  act_emb (2, 4, 192)

encode() mutates the dict in place and returns it: True


### `predict`: one step of dynamics

Given the first 3 frames and the actions taken, predict what comes next.

In [7]:
with torch.no_grad():
    pred = model.predict(out["emb"][:, :3], out["act_emb"][:, :3])

print(f"emb[:, :3]  {tuple(out['emb'][:, :3].shape)}  ->  pred {tuple(pred.shape)}")
print("\nOutput t is the prediction of the embedding at time t+1.")
print("So pred[:, -1] is the model's guess for the frame it has never seen.")

emb[:, :3]  (2, 3, 192)  ->  pred (2, 3, 192)

Output t is the prediction of the embedding at time t+1.
So pred[:, -1] is the model's guess for the frame it has never seen.


### `rollout`: imagining several plans at once

This is where the `S` axis appears. The model is handed **S candidate action
sequences** from one shared starting observation, and imagines all of them in
parallel — which is exactly what a planner needs.

In [8]:
S, horizon = 5, 8          # 5 candidate plans, 8 steps each
H = 3                      # 3 real observed frames

info = {
    # (B, S, H, C, H, W) -- the same start, repeated for each candidate
    "pixels": torch.rand(B, S, H, 3, 224, 224),
}
actions = torch.randn(B, S, horizon, A)

with torch.no_grad():
    out = model.rollout(info, actions, history_size=H)

print(f"start frames   {tuple(info['pixels'].shape)}")
print(f"action plans   {tuple(actions.shape)}   ({S} candidates, {horizon} steps)")
print(f"imagined       {tuple(out['predicted_emb'].shape)}")
print(f"\nThe model encoded {H} real frames, then imagined "
      f"{out['predicted_emb'].shape[2] - H} more without seeing any image.")

start frames   (2, 5, 3, 3, 224, 224)
action plans   (2, 5, 8, 10)   (5 candidates, 8 steps)
imagined       (2, 5, 9, 192)

The model encoded 3 real frames, then imagined 6 more without seeing any image.


### `criterion`: scoring the plans

Each imagined endpoint is compared to the goal embedding. The planner picks the
lowest cost.

In [9]:
out["goal_emb"] = torch.randn(B, S, horizon, D)

with torch.no_grad():
    cost = model.criterion(out)

print(f"cost {tuple(cost.shape)}  = one number per (batch, candidate plan)\n")
for b in range(B):
    row = "  ".join(f"{c:8.1f}" for c in cost[b])
    print(f"  batch {b}: {row}   -> best = plan {cost[b].argmin().item()}")

cost (2, 5)  = one number per (batch, candidate plan)

  batch 0:    258.2     220.3     192.7     193.4     182.9   -> best = plan 4
  batch 1:    191.1     248.9     205.0     238.0     187.3   -> best = plan 4


That `argmin` is the entire planning algorithm: imagine every plan, keep the
one that lands closest to the goal. Notebook 03 runs this against the real
environment with a trained model.

## Export

After editing this notebook, regenerate `lewm/jepa.py` from the repository root:

```bash
uv run nbdev-export
```

In [10]:
# Export explicitly from the repository root: uv run nbdev-export